## For Drive

In [1]:
"""from google.colab import drive
drive.mount('/content/drive')"""


"from google.colab import drive\ndrive.mount('/content/drive')"

## For Local

In [2]:
"""import kagglehub

path = kagglehub.dataset_download(
    "bhavikjikadara/dog-and-cat-classification-dataset",
    output_dir=r"D:\Code With Harry\Intro-To-Deeplearing\PYTORCH\5) Simple CNN\dataset"
)

print("Path to dataset files:", path)"""

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Dilip Das\AppData\Local\Temp\ipykernel_11024\231676676.py:1: SyntaxWarning: invalid escape sequence '\C'
  """import kagglehub


'import kagglehub\n\npath = kagglehub.dataset_download(\n    "bhavikjikadara/dog-and-cat-classification-dataset",\n    output_dir=r"D:\\Code With Harry\\Intro-To-Deeplearing\\PYTORCH\x05) Simple CNN\\dataset"\n)\n\nprint("Path to dataset files:", path)'

In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [4]:
import os


from PIL import Image

from torchvision import transforms

from sklearn.model_selection import train_test_split

In [5]:
import random 
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [6]:
def create_dataset(dataset_dir):

    images = []
    labels = []

    # if we have n classes, it will arrange them alphabetically and Give labels (apple-0, ball-2,...)
    class_names = sorted(os.listdir(dataset_dir))

    # getting the Path/ directry of Cat or Dog folder (folder/dataset/petdata/ dog or Cat)
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_dir, class_name)

        ## getting the path of each images (folder/dataset/petdata/ dog or Cat/ image1,image2...)
        for image_name in os.listdir(class_dir):
            image_path = os.path.join(class_dir, image_name)

            images.append(image_path)
            labels.append(label)
    return images, labels

In [7]:
class catdog(Dataset):

    def __init__(self, images, labels, transform = None):

        self.transform = transform


        self.images = images
        self.labels = labels


    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        label = self.labels[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform :
            image = self.transform(image)

        return image, label



In [8]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(90),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

In [9]:
root_dir = "../sample data/dataset/PetImages"
X, y = create_dataset(root_dir)

print(len(X))
print(len(y))

24998
24998


In [10]:
print(set(y))

{0, 1}


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [12]:
train_dataset = catdog(X_train, y_train, transform= train_transform)
test_dataset = catdog(X_test, y_test, transform= test_transform)


In [13]:
len(train_dataset)
len(test_dataset)

5000

In [14]:
#help(DataLoader)

In [15]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle= False)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

Device is cuda


#### Previous Results (with out Drop Out and batch Norm)

| Rotation | Epochs | Final Loss | Train Accuracy | Test Accuracy |
|----------|--------|------------|----------------|---------------|
| 10°      | 5      | 0.3614     | 85.25%         | ~84.64        |
| 10°      | 10     | 0.2736     | 89.22%         | 86.48%        |
| 90°      | 10     | 0.3987     | 81.16%         | 81.94%        |

#### Results (with Dropout and BatchNorm)

| Dropout | BatchNorm | Rotation | Epochs | Final Loss | Train Accuracy | Test Accuracy |
|---------|-----------|----------|--------|------------|----------------|---------------|
| 0.1     | Yes       | 90°      | 10     | 0.4892     | 78.32%         | 79.64%        |
| 0.1     | Yes       | 90°      | 15     | 0.4382     | 80.41%         | 82.52%        |

In [17]:
class CNN(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels= input_features, out_channels=32, kernel_size=3, padding=0),  #input channels --> RGB, but our input size is 256, got as reshape in 256*256
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(in_channels=32, out_channels= 64, kernel_size=3, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),


            nn.Conv2d(in_channels=64, out_channels= 128, kernel_size=3, padding=0),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(30*30*128,128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(64,1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        

        return x

In [18]:
images, labels = next(iter(train_dataloader))
print(images.shape)

model = CNN(input_features= images.shape[1])
model.to(device)

torch.Size([32, 3, 256, 256])


CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=115200, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=

In [19]:
# print(model)

In [20]:
#! pip install torchinfo

In [21]:
from torchinfo import summary
summary(model, input_size=(32, 3, 256, 256))

Layer (type:depth-idx)                   Output Shape              Param #
CNN                                      [32, 1]                   --
├─Sequential: 1-1                        [32, 128, 30, 30]         --
│    └─Conv2d: 2-1                       [32, 32, 254, 254]        896
│    └─BatchNorm2d: 2-2                  [32, 32, 254, 254]        64
│    └─ReLU: 2-3                         [32, 32, 254, 254]        --
│    └─MaxPool2d: 2-4                    [32, 32, 127, 127]        --
│    └─Conv2d: 2-5                       [32, 64, 125, 125]        18,496
│    └─BatchNorm2d: 2-6                  [32, 64, 125, 125]        128
│    └─ReLU: 2-7                         [32, 64, 125, 125]        --
│    └─MaxPool2d: 2-8                    [32, 64, 62, 62]          --
│    └─Conv2d: 2-9                       [32, 128, 60, 60]         73,856
│    └─BatchNorm2d: 2-10                 [32, 128, 60, 60]         256
│    └─ReLU: 2-11                        [32, 128, 60, 60]         --
│   

#### to calculate Flatte Size

In [22]:
"""dummy = torch.zeros(1, 3, 256, 256)
output = model.features(dummy)
print(output.shape)"""

'dummy = torch.zeros(1, 3, 256, 256)\noutput = model.features(dummy)\nprint(output.shape)'

In [23]:
#model.parameters()

In [24]:
for batch_features, batch_labels in train_dataloader:
    print(batch_labels.shape)
    break

torch.Size([32])


output from is CNN model is [32,1]
but here labels are only [32], so we have to add another......
so we have to unsqeeze(1)

In [25]:
learning_rate = 0.001
epochs = 15
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [26]:
for epoch in range(epochs):
    total_epochs_loss = 0
    for batch_features , batch_labels in train_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.float().unsqueeze(1).to(device)

        output = model(batch_features)

        loss = criterion(output, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epochs_loss = total_epochs_loss + loss.item()
    avg_loss = total_epochs_loss/len(train_dataloader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


d:\Ana\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch: 1 , Loss: 0.7084762321949005
Epoch: 2 , Loss: 0.6105909189224243
Epoch: 3 , Loss: 0.5893361998558044
Epoch: 4 , Loss: 0.5741085530281067
Epoch: 5 , Loss: 0.5513129087924957
Epoch: 6 , Loss: 0.5346987553596496
Epoch: 7 , Loss: 0.5174500836372375
Epoch: 8 , Loss: 0.5073365321874619
Epoch: 9 , Loss: 0.49478693680763247
Epoch: 10 , Loss: 0.4892078275680542
Epoch: 11 , Loss: 0.474859952712059
Epoch: 12 , Loss: 0.46385987482070923
Epoch: 13 , Loss: 0.4550575830936432
Epoch: 14 , Loss: 0.44591873977184293
Epoch: 15 , Loss: 0.43821453704833985


In [27]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_images, batch_labels in train_dataloader:

        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_images)

        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).long().squeeze(1)

        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

accuracy = correct / total

print(f"Training Accuracy: {accuracy * 100:.2f}%")

Training Accuracy: 80.41%


In [28]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_images, batch_labels in test_dataloader:

        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_images)

        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).long().squeeze(1)

        correct += (predictions == batch_labels).sum().item()
        total += batch_labels.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 82.52%
